In [ ]:
# install packages (if need be)
!pip install palmerpenguins --quiet
!pip install pytorch_lightning --quiet

# Prep

* Installing the packages (above) might take a few minutes
* Hereafter, you are advised to *connect* to a T4 GPU runtime (upper-right corner)

In [ ]:
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import sympy
import torch
import torch.nn as nn
import torch.nn.functional as F

from palmerpenguins import load_penguins
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sympy import isprime
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# seed function for reproducibility
def random_seed(seed_value):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    #random.seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

lr = 0.01 #learning rate

In [ ]:
# data loading
penguins = load_penguins()

# removing missing data
penguins_subset = penguins[['species', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']].dropna()

# Example 1

* two numerical explanatory variables
* response variable: `species`

In [ ]:
# data wrangling
# explanatory variables
X = penguins_subset[['flipper_length_mm', 'body_mass_g']]
X = StandardScaler().fit_transform(X) #min-max rescaling
X = torch.from_numpy(X).float() #convert to tensor, and ensure float data type

# response variable
y = penguins_subset['species'] #categorical reponse
y = LabelEncoder().fit_transform(y) #assigns integer to each category
y = torch.from_numpy(y).long() #convert to tensor, and ensure integer data type

# data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,  random_state=301)

# data loaders
batch_size = 8
train_loaders = DataLoader(TensorDataset(X_train, y_train), batch_size)
test_loaders = DataLoader(TensorDataset(X_test, y_test), batch_size)

In [ ]:
class FCN(pl.LightningModule):
  # Fully-Connected Network
  # assumes no hidden layer (i.e. going directly into activation functions)

    def __init__(self, input_layer_size, output_layer_size):
        super().__init__()
        self.input_layer_size = input_layer_size #input layer size (number of explanatory variables)
        self.output_layer_size = output_layer_size #output layer size (number of labels in response variable)
        self.fc1 = nn.Linear(input_layer_size, output_layer_size)
        self.test_step_outputs = []

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return x

    def training_step(self, batch, batch_idx): # what happen in each train step
        x, y = batch
        output = self(x)
        loss = F.cross_entropy(output, y)
        # self.log('train_loss', loss, on_epoch=True) # use this for logging (e.g. using TensorBoard)
        return {'loss':loss}

    def test_step(self, batch, batch_idx): # what happen in each test step
        x, y = batch
        output = self(x)
        loss = F.cross_entropy(output, y)
        self.test_step_outputs.append(loss)
        return {'loss':loss}

    def on_test_epoch_end(self):
        epoch_average = torch.stack(self.test_step_outputs).mean()
        self.log("test_epoch_average", epoch_average)
        self.test_step_outputs.clear()  # free memory

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=lr)
        return optimizer

In [ ]:
random_seed(301)
model = FCN(2,3)
trainer = pl.Trainer(max_epochs = 100)
trainer.fit(model, train_loaders)

In [ ]:
trainer.test(model, test_loaders)

In [ ]:
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

# Example 2: Prime Numbers



In [ ]:
df_prime = pd.DataFrame(list(range(8,101)), columns = ['x'])

df_prime['div2'] = np.where(df_prime['x'] % 2 == 0, 1, 0)
df_prime['div3'] = np.where(df_prime['x'] % 3 == 0, 1, 0)
df_prime['div5'] = np.where(df_prime['x'] % 5 == 0, 1, 0)
df_prime['div7'] = np.where(df_prime['x'] % 7 == 0, 1, 0)
#df_prime['is_prime'] = df_prime['x'].transform(lambda x:
#                                               np.where(sympy.isprime(x), 1, 0))
df_prime['is_prime'] = df_prime['x'].transform(lambda x:  sympy.isprime(x))

print(df_prime)

In [ ]:
# explanatory variables
X = df_prime[['div2', 'div3', 'div5', 'div7']]
X = X.to_numpy()
X = torch.from_numpy(X).float() #convert to tensor, and ensure float data type

# response variable
y = df_prime['is_prime'] #categorical reponse
y = LabelEncoder().fit_transform(y) #assigns integer to each category
y = torch.from_numpy(y).long() #convert to tensor, and ensure integer data type

# data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=301)

# data loaders
batch_size = 8
train_loaders = DataLoader(TensorDataset(X_train, y_train), batch_size)
test_loaders = DataLoader(TensorDataset(X_test, y_test), batch_size)

In [ ]:
random_seed(301)
model = FCN(4,2)
trainer = pl.Trainer(max_epochs = 100)
trainer.fit(model, train_loaders)

In [ ]:
trainer.test(model, test_loaders)

In [ ]:
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))